In [1]:
!pip install sentence-transformers faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import time
import numpy as np
import faiss

BASE = '/content/drive/MyDrive/semantic-search-system'

with open(f'{BASE}/data/processed/clean_corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

texts = corpus['texts']
print(f'Loaded {len(texts)} documents from Drive')

Mounted at /content/drive
Loaded 13574 documents from Drive


In [3]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-base-en-v1.5'
print(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)
DIM = model.get_sentence_embedding_dimension()
print(f'Embedding dimension: {DIM}')

Loading model: BAAI/bge-base-en-v1.5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 768


In [4]:
print(f'Encoding {len(texts)} documents...')
start = time.time()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f'Done in {time.time()-start:.1f}s')
print(f'Shape: {embeddings.shape}')

Encoding 13574 documents...


Batches:   0%|          | 0/213 [00:00<?, ?it/s]

Done in 231.8s
Shape: (13574, 768)


In [5]:
index = faiss.IndexFlatIP(DIM)
index.add(embeddings.astype(np.float32))
print(f'FAISS index built with {index.ntotal} vectors')

FAISS index built with 13574 vectors


In [6]:
np.save(f'{BASE}/models/embeddings.npy', embeddings)
faiss.write_index(index, f'{BASE}/models/faiss.index')

print(f'Saved: models/embeddings.npy')
print(f'Saved: models/faiss.index')
print(f'Sizes:')
print(f'  embeddings.npy : {os.path.getsize(f"{BASE}/models/embeddings.npy")/1e6:.1f} MB')
print(f'  faiss.index    : {os.path.getsize(f"{BASE}/models/faiss.index")/1e6:.1f} MB')

Saved: models/embeddings.npy
Saved: models/faiss.index
Sizes:
  embeddings.npy : 41.7 MB
  faiss.index    : 41.7 MB


In [7]:
test_queries = [
    'symptoms of flu and fever medication',
    'space shuttle launch nasa',
    'graphics card driver installation windows',
]

for q in test_queries:
    q_emb = model.encode([q], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, k=3)
    print(f'\nQuery: {q}')
    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0]), 1):
        print(f'  #{rank} score={score:.3f} [{corpus["categories"][idx]}]')


Query: symptoms of flu and fever medication
  #1 score=0.665 [sci.med]
  #2 score=0.645 [sci.med]
  #3 score=0.643 [sci.med]

Query: space shuttle launch nasa
  #1 score=0.707 [sci.space]
  #2 score=0.680 [sci.space]
  #3 score=0.671 [sci.space]

Query: graphics card driver installation windows
  #1 score=0.724 [comp.os.ms-windows.misc]
  #2 score=0.711 [comp.sys.ibm.pc.hardware]
  #3 score=0.709 [comp.os.ms-windows.misc]
